In [36]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()

True

In [37]:
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=200)

In [38]:
pdf_dir = Path("./docs")
pdf_paths = list(pdf_dir.glob("*.pdf"))

documents = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

In [39]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=20,
    add_start_index=True,
)

chunks = text_splitter.split_documents(documents)

In [40]:
db = Chroma.from_documents(chunks, embedding=embeddings_model, persist_directory="./data/chroma_db")

In [41]:
vectordb = Chroma(persist_directory="./data/chroma_db", embedding_function=embeddings_model)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Use the following context to answer the question. "
     "If the answer is not in the context, say you don't know. Be concise.\n\n"
     "{context}"),
    ("human", "{input}"),
])

rag_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))


def ask(question: str) -> str:
    return rag_chain.invoke({"input": question})["answer"]

In [42]:
user_question = input("Ask a question about the PDFs: ")
answer = ask(user_question)
print("Answer:", answer)

Answer: Frey é amiga de Vayne.
